## Type Coercion

Was passiert, wenn der übergebene Wert nicht genau dem annotierten Typ entspricht, etwa `0` für ein `float`-Field oder der String `"1.2"`? Pydantic wandelt passende Werte um (Coercion) und lehnt unpassende ab.

**Danach ist klar:**
- welche Werte Pydantic im Standardmodus (Lax Mode) umwandelt,
- welche Werte abgelehnt werden, etwa eine Zahl für ein `str`-Field,
- warum die Validierung hilft, wenn sich das Format von Daten ändert.

Voraussetzung: Notebook `04 - Serialization`.

In [ ]:
# `BaseModel` ist die Basisklasse für alle Models, `ValidationError` wird später gebraucht, um Fehlerfälle abzufangen.
from pydantic import BaseModel, ValidationError

In [ ]:
# `Coordinates` verlangt zwei Pflicht-Fields, beide vom Typ `float`.
class Coordinates(BaseModel):
    x: float
    y: float

In [ ]:
# Hier passen beide Werte bereits zum Typ `float`, es gibt also nichts umzuwandeln.
# Die letzte Zeile gibt die Instanz aus.
p1 = Coordinates(x=1.1, y=2.2)
p1

In [ ]:
# `model_fields` zeigt, welchen Typ Pydantic pro Field erwartet: zweimal `float`.
Coordinates.model_fields

In [ ]:
# Ein Blick auf den Typ bestätigt, dass `x` tatsächlich ein `float` ist.
type(p1.x)

**Vorhersage:** `x` bekommt den `int` `0`, `y` den String `"1.2"`. Löst das einen Fehler aus, und welche Werte stehen danach in der Instanz?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Pydantic wandelt beim Erzeugen um (Coercion): `0` wird zu `0.0`, `"1.2"` zu `1.2`. Beide Werte passen in `float`, deshalb gibt es keinen Fehler. Das ist der Lax Mode, die Standardeinstellung.
</details>

In [ ]:
p2 = Coordinates(x=0, y="1.2")
p2

In [ ]:
# Obwohl ein `int` und ein `str` übergeben wurden, sind beide Fields jetzt `float`.
type(p2.x), type(p2.y)

### Wo Coercion endet

Wird alles umgewandelt? Eine Zahl für ein `str`-Field zeigt eine Grenze.

In [ ]:
# Zum Vergleich ein Model mit einem einzigen `str`-Field.
class Model(BaseModel):
    field: str

In [ ]:
# Ein `str` passt direkt zum Feldtyp, deshalb muss nichts umgewandelt werden.
Model(field="python")

**Vorhersage:** `100` ist ein `int`. Wandelt Pydantic es in den String `"100"` um?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Nein: Im Lax Mode wird eine Zahl nicht automatisch zum `str`. Die `ValidationError` nennt den Fehlertyp `string_type`.

Hinweis: Mit der Konfiguration `coerce_numbers_to_str` lässt sich das Verhalten ändern.

Kurz gesagt: Pydantic wandelt nur bestimmte Typen ineinander um, nicht alle.
</details>

In [ ]:
try:
    Model(field=100)
except ValidationError as exc:
    print(exc)

### Veränderte Datenformate

Was passiert, wenn eine API ihr Format ändert? Die Validierung bemerkt das sofort.

**Vorhersage:** In den neuen Daten ist `email` ein Objekt statt eines Strings. Wandelt Pydantic es um, oder schlägt die Validierung fehl?

In [ ]:
# `Contact` erwartet `email` als einfachen String.
class Contact(BaseModel):
    email: str

In [ ]:
# So sah das ursprüngliche Format aus: `email` ist ein String.
initial_json_data = '''
{
    "email": "inewton@principia.com"
}
'''

In [ ]:
# Der JSON-Text passt zum Model, die Instanz entsteht ohne Fehler.
Contact.model_validate_json(initial_json_data)

In [ ]:
# In der neuen Version der Daten ist `email` plötzlich ein Objekt mit `personal` und `work`.
new_json_data = '''
{
    "email": {
        "personal": "inewton@principia.com",
        "work": "isaac.newton@themint.com"
    }
}
'''

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `email` ist jetzt ein `dict`, kein `str`. Dafür gibt es keine Coercion, deshalb meldet Pydantic `string_type`. Die Meldung nennt Feld, Fehlertyp und den übergebenen Input.

Kurz gesagt: Ändert sich das Datenformat, schlägt die Validierung sofort fehl, statt falsche Daten stillschweigend zu übernehmen.
</details>

In [ ]:
try:
    Contact.model_validate_json(new_json_data)
except ValidationError as exc:
    print(exc)

### Zusammenfassung

- Im Lax Mode wandelt Pydantic passende Werte um: `0` wird zu `0.0`, `"1.2"` zu `1.2`, wenn das Field `float` erwartet.
- Eine Zahl für ein `str`-Field wird abgelehnt (`string_type`). `coerce_numbers_to_str` ändert das.
- Auch ein `dict` für ein `str`-Field wird abgelehnt. Geänderte Datenformate fallen so sofort auf.
- `model_fields` zeigt die Typen, an denen sich die Coercion orientiert.

Weiter mit `06 - Required vs Optional Fields`.